
# DATA 304 — Module 10: Dates and Times 

This notebook demonstrates:
- Parsing and normalizing timestamps
- Extracting date features
- Time arithmetic and corrections
- Handling time zones and DST
- Cleaning and validating temporal data
- Aggregation, resampling, grouping, rolling metrics
- Temporal joins (`merge_asof`)


## Setup

In [ ]:

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo  # Python 3.9+
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## Create sample data

In [ ]:

# Synthetic transactions with mixed date formats
raw_dates = [
    "2025-10-20", "20/10/2025", "Oct 21, 2025", "2025/10/22", "10-23-2025", "invalid",
    "2025-10-24 10:15:00", "2025-10-24 10:15:00"  # duplicate
]
sales = [100, 120, 130, 90, 80, 75, 200, 200]

transactions = pd.DataFrame({"raw_date": raw_dates, "sales": sales})
transactions


## Parsing strings → datetimes

In [ ]:

# Flexible parser with error coercion
transactions["timestamp"] = pd.to_datetime(transactions["raw_date"], errors="coerce")
transactions[["raw_date", "timestamp", "sales"]]


### When you know the exact format

In [ ]:

# Example of explicit format (day-first)
ex = pd.Series(["20/10/2025", "21/10/2025", "22/10/2025"])
parsed = pd.to_datetime(ex, format="%d/%m/%Y")
pd.DataFrame({"original": ex, "parsed": parsed})


## Extracting date features

In [ ]:

parsed_ok = transactions.dropna(subset=["timestamp"]).copy()
parsed_ok["year"] = parsed_ok["timestamp"].dt.year
parsed_ok["month"] = parsed_ok["timestamp"].dt.month
parsed_ok["day"] = parsed_ok["timestamp"].dt.day
parsed_ok["weekday"] = parsed_ok["timestamp"].dt.weekday  # Monday=0
parsed_ok["hour"] = parsed_ok["timestamp"].dt.hour
parsed_ok[["timestamp", "year", "month", "day", "weekday", "hour", "sales"]]


## Time arithmetic and corrections

In [ ]:

dt = datetime(2025, 10, 29, 8, 0, 0)
next_day = dt + timedelta(days=1)
gap = datetime.now() - dt
dt_fixed = dt.replace(hour=10, minute=30)

print("dt:", dt)
print("next_day:", next_day)
print("gap total seconds:", int(gap.total_seconds()))
print("dt_fixed:", dt_fixed)


## Handling time zones and DST

In [ ]:
tz_demo = pd.DataFrame({
    "local_time": pd.to_datetime([
        "2025-03-09 01:30:00",  # spring forward: nonexistent local time
        "2025-11-02 01:30:00"   # fall back: ambiguous local time
    ])
})

# Boolean mask: which rows are the ambiguous fallback time?
amb_mask = tz_demo["local_time"].dt.strftime("%Y-%m-%d %H:%M:%S").eq("2025-11-02 01:30:00")

tz_demo["eastern"] = tz_demo["local_time"].dt.tz_localize(
    "US/Eastern",
    ambiguous=amb_mask,           # True for rows treated as DST; False for standard time
    nonexistent="shift_forward"   # resolve spring-forward gaps
)
tz_demo["utc"] = tz_demo["eastern"].dt.tz_convert("UTC")
tz_demo


In [ ]:
spring = datetime(2025, 3, 9, 1, 30, tzinfo=ZoneInfo("US/Eastern"))
fall_1 = datetime(2025, 11, 2, 1, 30, tzinfo=ZoneInfo("US/Eastern"))
fall_2 = datetime(2025, 11, 2, 1, 30, tzinfo=ZoneInfo("US/Eastern"))
fall_2 = fall_2.replace(fold=1)  # standard-time instance

print("Spring forward:", spring, "→", spring.astimezone(ZoneInfo("UTC")))
print("Fall back (DST):", fall_1, "→", fall_1.astimezone(ZoneInfo("UTC")))
print("Fall back (Standard):", fall_2, "→", fall_2.astimezone(ZoneInfo("UTC")))

## Cleaning timestamps

In [ ]:

clean = transactions.copy()
# Remove duplicates on timestamp after parsing; sort
clean["timestamp"] = pd.to_datetime(clean["raw_date"], errors="coerce")
clean = clean.drop_duplicates("timestamp").sort_values("timestamp")

# Fill missing timestamps by forward-fill on sales (demo purposes)
clean["sales"] = clean["sales"].astype(float)
clean["sales_ffill"] = clean["sales"].ffill()

# Validate range
mask = clean["timestamp"].between("2000-01-01", "2030-12-31")
clean = clean.loc[mask]
clean[["raw_date", "timestamp", "sales", "sales_ffill"]]


## Aggregation and resampling

In [ ]:

# Build a higher-frequency example
rng = pd.date_range("2025-10-01", periods=24*7, freq="h")  # hourly for a week
np.random.seed(0)
values = np.random.randint(80, 140, size=len(rng))

series = pd.DataFrame({"timestamp": rng, "value": values}).set_index("timestamp")

daily_mean = series.resample("D").mean()
weekly_sum = series.resample("W").sum()

print("Daily mean:")
display(daily_mean.head())

print("Weekly sum:")
display(weekly_sum)

# Plot daily mean (single plot, matplotlib, no explicit colors)
daily_mean.plot(title="Daily Mean")
plt.xlabel("Date")
plt.ylabel("Value")
plt.show()


## Upsampling and interpolation

In [ ]:

# Downsample hourly → 3-hourly to simulate missing points, then upsample back
down = series.resample("3h").mean()
up = down.resample("h").asfreq()  # introduce NaNs
interp = up.interpolate()

print("Downsampled (3H):")
display(down.head())
print("Upsampled (H) with NaNs:")
display(up.head(10))
print("Interpolated:")
display(interp.head(10))

interp.plot(title="Upsampled then Interpolated")
plt.xlabel("Date")
plt.ylabel("Value")
plt.show()


## Temporal grouping and rolling metrics

In [ ]:

monthly_sum = series.groupby(pd.Grouper(freq="ME"))["value"].sum()
rolling_7d = series.rolling("7D")["value"].mean()

print("Monthly sum:")
display(monthly_sum.head())

ax = series["value"].plot(alpha=0.5, title="Rolling 7-Day Mean")
rolling_7d.plot(ax=ax)
plt.xlabel("Date")
plt.ylabel("Value")
plt.show()


## Temporal joins (`merge_asof`)

In [ ]:

# df1: sparse events; df2: more frequent reference series
df1 = pd.DataFrame({
    "timestamp": pd.to_datetime(["2025-10-01 10:00:01", "2025-10-01 10:00:05", "2025-10-01 10:02:10"]),
    "event": ["A", "B", "C"]
}).sort_values("timestamp")

df2 = pd.DataFrame({
    "timestamp": pd.to_datetime(["2025-10-01 10:00:00", "2025-10-01 10:00:06", "2025-10-01 10:02:00", "2025-10-01 10:03:00"]),
    "price": [101.5, 101.7, 101.4, 101.9]
}).sort_values("timestamp")

merged = pd.merge_asof(df1, df2, on="timestamp", direction="nearest", tolerance=pd.Timedelta("2s"))
merged


## Validation checks

In [ ]:

# Monotonicity and duplicates
is_mono = series.index.is_monotonic_increasing
has_dups = series.index.duplicated().any()
print("Monotonic increasing index:", is_mono)
print("Any duplicate timestamps:", has_dups)

# Frequency inference
freq = pd.infer_freq(series.index)
print("Inferred frequency:", freq)

# Interval sanity
interval_demo = pd.DataFrame({
    "start_time": pd.to_datetime(["2025-10-01 10:00", "2025-10-01 11:00"]),
    "end_time":   pd.to_datetime(["2025-10-01 10:30", "2025-10-01 10:45"])  # second one invalid (end < start)
})
interval_demo.assign(valid=(interval_demo["start_time"] < interval_demo["end_time"]))
